In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import logging
import sys

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s',
    stream=sys.stdout
)


In [2]:
import pandas as pd
import numpy as np
import os
sys.path.append('../core')
import config
from feature_extraction import ClinicalNLP_Pipeline
from preprocessing import load_and_merge_data, preprocess_multimodal_data

INFO: NumExpr defaulting to 8 threads.


C:\Users\damitha\.conda\envs\irp_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
raw_data_path = config.raw_data_path
processed_data_path = config.processed_data_path


# Data Integration & Preprocessing

In [4]:
raw_notes_df = pd.read_csv(nlp_features_path)

NameError: name 'nlp_features_path' is not defined

In [6]:
print("Input Columns:", raw_notes_df.columns.tolist())

Input Columns: ['hadm_id', 'text']


In [7]:
nlp_pipeline = ClinicalNLP_Pipeline()

INFO: NLP Pipeline Initialized on: cuda:0 (ID: 0)


c:\Users\damitha\.conda\envs\irp_gpu\lib\site-packages\huggingface_hub-0.29.2-py3.8.egg\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
c:\Users\damitha\.conda\envs\irp_gpu\lib\site-packages\transformers\utils\generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [8]:
notes_enhanced_df = nlp_pipeline.process_dataframe(raw_notes_df, text_col='text')

INFO: Extracting features from 27677 notes on cuda:0...


Extracting NLP Features:   0%|          | 9/27677 [00:00<09:46, 47.19it/s]c:\Users\damitha\.conda\envs\irp_gpu\lib\site-packages\transformers\pipelines\base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Extracting NLP Features: 100%|██████████| 27677/27677 [08:28<00:00, 54.43it/s]


In [10]:
output_path = os.path.join(config.processed_data_path, 'nlp_features.csv')

In [12]:
notes_enhanced_df[['hadm_id', 'nlp_sentiment_score', 'nlp_uncertainty_score']].to_csv(output_path, index=False)
print(f"Success! Features saved to {output_path}")
print("New Columns:", notes_enhanced_df.columns.tolist())

Success! Features saved to E:\IIT\IRP\data\processed\nlp_features.csv
New Columns: ['hadm_id', 'text', 'nlp_uncertainty_score', 'nlp_sentiment_score']


merge strucutred data with notes

In [5]:
structured_path = os.path.join(raw_data_path, 'ami_cohort_structured_features.csv')
nlp_features_path = os.path.join(processed_data_path, 'nlp_features.csv')

In [6]:
from preprocessing import load_and_merge_data, preprocess_multimodal_data

In [7]:

df_final = load_and_merge_data(structured_path, nlp_features_path)

INFO: Loading datasets...
INFO: Merged Data Shape: (27677, 27)


In [8]:
import pickle
from sklearn.preprocessing import StandardScaler

# 1. Grab just the three columns your Streamlit app needs to scale
# (Make sure 'df' is the name of your dataframe in the notebook)
vitals_to_scale = df_final[['admission_age', 'Troponin_T_max', 'CK_MB_max']]

# 2. Train a standalone scaler just for the Streamlit dashboard
vitals_scaler = StandardScaler()
vitals_scaler.fit(vitals_to_scale)

# 3. Save it as a .pkl file
with open('../app/models/vitals_scaler.pkl', 'wb') as f:
    pickle.dump(vitals_scaler, f)
    
print("vitals_scaler.pkl saved successfully! Move this to your app's 'models' folder.")

vitals_scaler.pkl saved successfully! Move this to your app's 'models' folder.


In [12]:
X, y, feature_names = preprocess_multimodal_data(df_final)

INFO: Preprocessing data (Imputation + Normalization)...
INFO: Preprocessing complete.


In [13]:
print(feature_names)

['admission_age', 'Troponin_T_max', 'CK_MB_max', 'nlp_sentiment_score', 'nlp_uncertainty_score', 'gender_F', 'gender_M', 'admission_type_AMBULATORY OBSERVATION', 'admission_type_DIRECT EMER.', 'admission_type_DIRECT OBSERVATION', 'admission_type_ELECTIVE', 'admission_type_EU OBSERVATION', 'admission_type_EW EMER.', 'admission_type_OBSERVATION ADMIT', 'admission_type_SURGICAL SAME DAY ADMISSION', 'admission_type_URGENT', 'insurance_Medicaid', 'insurance_Medicare', 'insurance_No charge', 'insurance_Other', 'insurance_Private']


In [15]:
X_path = os.path.join(processed_data_path, 'X_data.npy')
y_path = os.path.join(processed_data_path, 'y_data.npy')
feature_names_path = os.path.join(processed_data_path, 'feature_names.npy')

In [16]:
np.save(X_path, X)
np.save(y_path, y)
np.save(feature_names_path, feature_names)